# AgingClockBench — Custom Data

This notebook shows how to run AgingClockBench on **your own CSV data**.

Your CSV must contain the columns listed below (all in standard NHANES clinical units).
Mortality columns (`vital_status`, `followup_months`) are optional — add them to unlock Cox PH hazard ratios.

In [ ]:
import pandas as pd
from agingclockbench import PhenoAge, KDM, DunedinPACEProxy, BenchmarkSuite

## Required columns

| Column | Unit | Clocks |
|--------|------|--------|
| `age` | years | All |
| `albumin_g_dl` | g/dL | PhenoAge, KDM, DunedinPACEProxy |
| `creatinine_mg_dl` | mg/dL | PhenoAge, KDM, DunedinPACEProxy |
| `glucose_mg_dl` | mg/dL | PhenoAge, KDM, DunedinPACEProxy |
| `crp_mg_l` | mg/L | PhenoAge |
| `lymphocyte_pct` | % | PhenoAge, KDM, DunedinPACEProxy |
| `mcv_fl` | fL | PhenoAge, KDM, DunedinPACEProxy |
| `rdw_pct` | % | PhenoAge, KDM, DunedinPACEProxy |
| `alp_u_l` | U/L | PhenoAge, KDM |
| `wbc_k_ul` | 10³/μL | PhenoAge, KDM, DunedinPACEProxy |
| `vital_status` *(optional)* | 0/1 | Cox PH |
| `followup_months` *(optional)* | months | Cox PH |

In [ ]:
# ── Replace with your file path ──
# df = pd.read_csv('my_cohort.csv')

# Demo: create a small synthetic dataset
import numpy as np
np.random.seed(42)
n = 200
ages = np.random.uniform(30, 80, n)
df = pd.DataFrame({
    'age':              ages,
    'albumin_g_dl':     np.clip(4.5 - 0.003*ages + np.random.normal(0, 0.3, n), 3.0, 5.5),
    'creatinine_mg_dl': np.clip(0.5 + 0.005*ages + np.random.normal(0, 0.2, n), 0.3, 2.0),
    'glucose_mg_dl':    np.clip(75 + 0.5*ages  + np.random.normal(0, 15, n),  60, 300),
    'crp_mg_l':         np.clip(np.exp(np.random.normal(0.5, 0.8, n)), 0.05, 50),
    'lymphocyte_pct':   np.clip(30 - 0.05*ages + np.random.normal(0, 6, n),  5, 60),
    'mcv_fl':           np.clip(88 + 0.04*ages  + np.random.normal(0, 4, n), 70, 110),
    'rdw_pct':          np.clip(12 + 0.015*ages + np.random.normal(0, 0.8, n), 10, 20),
    'alp_u_l':          np.clip(70 + 0.2*ages   + np.random.normal(0, 25, n), 20, 300),
    'wbc_k_ul':         np.clip(8 - 0.01*ages   + np.random.normal(0, 1.5, n), 2, 15),
})
print(f"Dataset: {len(df)} participants, age {df.age.min():.0f}–{df.age.max():.0f} yr")
df.head()

## Run the clocks

In [ ]:
results = {
    'PhenoAge':        PhenoAge().transform(df),
    'KDM':             KDM().transform(df),
    'DunedinPACEProxy': DunedinPACEProxy().transform(df),
}

for name, res in results.items():
    print(f"{name:20s}: BA = {res.biological_ages.mean():.1f} yr  "
          f"accel = {res.accel.mean():.1f} yr  "
          f"({res.missing_data_pct:.0f}% rows dropped)")

## Benchmark (no mortality data)

In [ ]:
# Without mortality columns, Cox HR will be NaN — that's expected
suite = BenchmarkSuite()
report = suite.run(df, results)
report.to_dataframe()

## Visualize

In [ ]:
import matplotlib
%matplotlib inline
fig = report.plot_comparison()
fig.savefig('comparison.png', dpi=120, bbox_inches='tight')

In [ ]:
fig2 = report.plot_correlation_heatmap()
fig2.savefig('clock_correlation.png', dpi=120, bbox_inches='tight')

## Export to HTML

In [ ]:
report.to_html('benchmark_report.html')
print('Open benchmark_report.html in your browser.')